In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import holidays
from datetime import date


CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

CREACION DIMENSION DATE

In [3]:
# dimensionDate = pd.DataFrame()
# dimensionDate.head()

TRANSFORMACION 

In [4]:
# dimensionDate["DateKey"] = None
# dimensionDate["FullDateAlternateKey"] = None
# dimensionDate["DayNumberOfWeek"]  = None
# dimensionDate["EnglishDayNameOfWeek"] = None
# dimensionDate["SpanishDayNameOfWeek"] = None
# dimensionDate["FrenchDayNameOfWeek"] = None
# dimensionDate["DayNumberOfMonth"] = None
# dimensionDate["DayNumberOfYear"] = None
# dimensionDate["WeekNumberOfYear"] = None
# dimensionDate["EnglishMonthName"] = None
# dimensionDate["SpanishMonthName"] = None
# dimensionDate["FrenchMonthName"] = None
# dimensionDate["MonthNumberOfYear"] = None 
# dimensionDate["CalendarQuarter"] = None
# dimensionDate["CalendarYear"] = None
# dimensionDate["CalendarSemester"] = None
# dimensionDate["FiscalQuarter"] = None
# dimensionDate["FiscalYear"] = None
# dimensionDate["FiscalSemester"] = None

# dimensionDate.head()

In [7]:

# Rango de fechas: desde 2005-01-01 hasta, por ejemplo, 2010-12-31
dates = pd.date_range(start="2005-01-01", end="2010-12-31", freq="D")

df = pd.DataFrame({"FullDateAlternateKey": dates})

# Clave numérica estilo YYYYMMDD
df["DateKey"] = df["FullDateAlternateKey"].dt.strftime("%Y%m%d").astype(int)

# Día de la semana (1=Monday, 7=Sunday según ISO, ajustamos si quieres que 7 sea sábado como tu ejemplo)
df["DayNumberOfWeek"] = df["FullDateAlternateKey"].dt.dayofweek + 1  # Monday=1 ... Sunday=7


# Mostrar primeras filas
# print(df.head(10))
df

,FullDateAlternateKey,DateKey,DayNumberOfWeek
0,2005-01-01,20050101,6
1,2005-01-02,20050102,7
2,2005-01-03,20050103,1
3,2005-01-04,20050104,2
4,2005-01-05,20050105,3
...,...,...,...
2186,2010-12-27,20101227,1
2187,2010-12-28,20101228,2
2188,2010-12-29,20101229,3
2189,2010-12-30,20101230,4


In [8]:

# Nombres de días
days_en = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
days_es = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
days_fr = ["Lundi","Mardi","Mercredi","Jeudi","Vendredi","Samedi","Dimanche"]

df["EnglishDayNameOfWeek"] = df["DayNumberOfWeek"].apply(lambda x: days_en[x-1])
df["SpanishDayNameOfWeek"] = df["DayNumberOfWeek"].apply(lambda x: days_es[x-1])
df["FrenchDayNameOfWeek"]  = df["DayNumberOfWeek"].apply(lambda x: days_fr[x-1])

# Día del mes y del año
df["DayNumberOfMonth"] = df["FullDateAlternateKey"].dt.day
df["DayNumberOfYear"]  = df["FullDateAlternateKey"].dt.dayofyear

# Semana del año
df["WeekNumberOfYear"] = df["FullDateAlternateKey"].dt.isocalendar().week
# print(df.head(10))
df

,FullDateAlternateKey,DateKey,DayNumberOfWeek,EnglishDayNameOfWeek,SpanishDayNameOfWeek,FrenchDayNameOfWeek,DayNumberOfMonth,DayNumberOfYear,WeekNumberOfYear
0,2005-01-01,20050101,6,Saturday,Sábado,Samedi,1,1,53
1,2005-01-02,20050102,7,Sunday,Domingo,Dimanche,2,2,53
2,2005-01-03,20050103,1,Monday,Lunes,Lundi,3,3,1
3,2005-01-04,20050104,2,Tuesday,Martes,Mardi,4,4,1
4,2005-01-05,20050105,3,Wednesday,Miércoles,Mercredi,5,5,1
...,...,...,...,...,...,...,...,...,...
2186,2010-12-27,20101227,1,Monday,Lunes,Lundi,27,361,52
2187,2010-12-28,20101228,2,Tuesday,Martes,Mardi,28,362,52
2188,2010-12-29,20101229,3,Wednesday,Miércoles,Mercredi,29,363,52
2189,2010-12-30,20101230,4,Thursday,Jueves,Jeudi,30,364,52


In [9]:

# Meses
months_en = ["January","February","March","April","May","June","July","August","September","October","November","December"]
months_es = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]
months_fr = ["Janvier","Février","Mars","Avril","Mai","Juin","Juillet","Août","Septembre","Octobre","Novembre","Décembre"]

df["MonthNumberOfYear"] = df["FullDateAlternateKey"].dt.month
df["EnglishMonthName"]  = df["MonthNumberOfYear"].apply(lambda x: months_en[x-1])
df["SpanishMonthName"]  = df["MonthNumberOfYear"].apply(lambda x: months_es[x-1])
df["FrenchMonthName"]   = df["MonthNumberOfYear"].apply(lambda x: months_fr[x-1])
# print(df.head(10))
df

,FullDateAlternateKey,DateKey,DayNumberOfWeek,EnglishDayNameOfWeek,SpanishDayNameOfWeek,FrenchDayNameOfWeek,DayNumberOfMonth,DayNumberOfYear,WeekNumberOfYear,MonthNumberOfYear,EnglishMonthName,SpanishMonthName,FrenchMonthName
0,2005-01-01,20050101,6,Saturday,Sábado,Samedi,1,1,53,1,January,Enero,Janvier
1,2005-01-02,20050102,7,Sunday,Domingo,Dimanche,2,2,53,1,January,Enero,Janvier
2,2005-01-03,20050103,1,Monday,Lunes,Lundi,3,3,1,1,January,Enero,Janvier
3,2005-01-04,20050104,2,Tuesday,Martes,Mardi,4,4,1,1,January,Enero,Janvier
4,2005-01-05,20050105,3,Wednesday,Miércoles,Mercredi,5,5,1,1,January,Enero,Janvier
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,2010-12-27,20101227,1,Monday,Lunes,Lundi,27,361,52,12,December,Diciembre,Décembre
2187,2010-12-28,20101228,2,Tuesday,Martes,Mardi,28,362,52,12,December,Diciembre,Décembre
2188,2010-12-29,20101229,3,Wednesday,Miércoles,Mercredi,29,363,52,12,December,Diciembre,Décembre
2189,2010-12-30,20101230,4,Thursday,Jueves,Jeudi,30,364,52,12,December,Diciembre,Décembre


In [10]:

# Trimestres y semestres calendario
df["CalendarQuarter"]  = df["FullDateAlternateKey"].dt.quarter
df["CalendarYear"]     = df["FullDateAlternateKey"].dt.year
df["CalendarSemester"] = ((df["MonthNumberOfYear"]-1)//6)+1
# print(df.head(10))
df


,FullDateAlternateKey,DateKey,DayNumberOfWeek,EnglishDayNameOfWeek,SpanishDayNameOfWeek,FrenchDayNameOfWeek,DayNumberOfMonth,DayNumberOfYear,WeekNumberOfYear,MonthNumberOfYear,EnglishMonthName,SpanishMonthName,FrenchMonthName,CalendarQuarter,CalendarYear,CalendarSemester
0,2005-01-01,20050101,6,Saturday,Sábado,Samedi,1,1,53,1,January,Enero,Janvier,1,2005,1
1,2005-01-02,20050102,7,Sunday,Domingo,Dimanche,2,2,53,1,January,Enero,Janvier,1,2005,1
2,2005-01-03,20050103,1,Monday,Lunes,Lundi,3,3,1,1,January,Enero,Janvier,1,2005,1
3,2005-01-04,20050104,2,Tuesday,Martes,Mardi,4,4,1,1,January,Enero,Janvier,1,2005,1
4,2005-01-05,20050105,3,Wednesday,Miércoles,Mercredi,5,5,1,1,January,Enero,Janvier,1,2005,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,2010-12-27,20101227,1,Monday,Lunes,Lundi,27,361,52,12,December,Diciembre,Décembre,4,2010,2
2187,2010-12-28,20101228,2,Tuesday,Martes,Mardi,28,362,52,12,December,Diciembre,Décembre,4,2010,2
2188,2010-12-29,20101229,3,Wednesday,Miércoles,Mercredi,29,363,52,12,December,Diciembre,Décembre,4,2010,2
2189,2010-12-30,20101230,4,Thursday,Jueves,Jeudi,30,364,52,12,December,Diciembre,Décembre,4,2010,2


In [12]:

# Supongamos que el año fiscal empieza en abril (puedes ajustar)
df["FiscalYear"]     = df["FullDateAlternateKey"].apply(lambda d: d.year if d.month>=4 else d.year-1)
df["FiscalQuarter"]  = ((df["FullDateAlternateKey"].dt.month-4)%12)//3 + 1
df["FiscalSemester"] = ((df["FiscalQuarter"]-1)//2)+1

# print(df.head(10))
df

,FullDateAlternateKey,DateKey,DayNumberOfWeek,EnglishDayNameOfWeek,SpanishDayNameOfWeek,FrenchDayNameOfWeek,DayNumberOfMonth,DayNumberOfYear,WeekNumberOfYear,MonthNumberOfYear,EnglishMonthName,SpanishMonthName,FrenchMonthName,CalendarQuarter,CalendarYear,CalendarSemester,FiscalYear,FiscalQuarter,FiscalSemester
0,2005-01-01,20050101,6,Saturday,Sábado,Samedi,1,1,53,1,January,Enero,Janvier,1,2005,1,2004,4,2
1,2005-01-02,20050102,7,Sunday,Domingo,Dimanche,2,2,53,1,January,Enero,Janvier,1,2005,1,2004,4,2
2,2005-01-03,20050103,1,Monday,Lunes,Lundi,3,3,1,1,January,Enero,Janvier,1,2005,1,2004,4,2
3,2005-01-04,20050104,2,Tuesday,Martes,Mardi,4,4,1,1,January,Enero,Janvier,1,2005,1,2004,4,2
4,2005-01-05,20050105,3,Wednesday,Miércoles,Mercredi,5,5,1,1,January,Enero,Janvier,1,2005,1,2004,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,2010-12-27,20101227,1,Monday,Lunes,Lundi,27,361,52,12,December,Diciembre,Décembre,4,2010,2,2010,3,2
2187,2010-12-28,20101228,2,Tuesday,Martes,Mardi,28,362,52,12,December,Diciembre,Décembre,4,2010,2,2010,3,2
2188,2010-12-29,20101229,3,Wednesday,Miércoles,Mercredi,29,363,52,12,December,Diciembre,Décembre,4,2010,2,2010,3,2
2189,2010-12-30,20101230,4,Thursday,Jueves,Jeudi,30,364,52,12,December,Diciembre,Décembre,4,2010,2,2010,3,2


CARGAR A LA BODEGA

In [14]:
df.to_sql('dimensionDate',motorBodegaDatos, if_exists='replace',index=False)

101